## Setup

In [30]:
import os
from dotenv import load_dotenv
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"].strip()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")   # for the caching demo
gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY2"].strip())  # for the tokenizer
print("✅ Ready")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Ready


## Step 1 — The Model Doesn't Read Words

Every prompt is first chopped into **tokens**. That's the unit the model actually processes.

In [31]:
phrase = "Acme was founded in 1998 in Austin, Texas."
n = gclient.models.count_tokens(model="gemini-2.5-flash-lite", contents=phrase).total_tokens

print(f'"{phrase}"')
print(f"→ {n} tokens")

"Acme was founded in 1998 in Austin, Texas."
→ 16 tokens


## Step 2 — Tokens Become Vectors (Q, K, V)

For every token, the model computes **three** vectors:

- **Q (Query)** — *what am I looking for?*
- **K (Key)** — *what am I about?*
- **V (Value)** — *what do I actually carry?*

Running attention over them (every token looking at every other) is the expensive `O(n²)` math. **That compute is what you pay for — not the text.**

**The key idea:** once the model computes these vectors, it **stores** them. So on a follow-up, your big knowledge base hasn't changed — same words, same tokens, already computed. The model just **loads it back** and only computes your **new question**. That's the **KV cache**: it isn't saving your text, it's saving the math it already did — so it never does it twice.

Those vectors live inside Gemini — we can't print them. But we **can** measure when they get reused: `usage_metadata` reports `cache_read`. That's our window into the cache.

In [32]:
import uuid
session = uuid.uuid4().hex[:8]   # fresh marker → this run's cache is isolated

def knowledge_base(first_word):
    return (
        f"[session {session}] {first_word} You are a support agent for Acme Corp.\n"
        + ("Acme was founded in 1998 in Austin, Texas. Employees get 20 vacation days. "
           "Returns are accepted within 30 days. Support hours are 9 to 5 Eastern. " * 350)
    )

def ask(first_word, question):
    msg = model.invoke(knowledge_base(first_word) + "\n\nQuestion: " + question)
    cached = (msg.usage_metadata.get("input_token_details") or {}).get("cache_read", 0)
    print(f"first word: {first_word!r:9}  input: {msg.usage_metadata['input_tokens']:>6}  from cache: {cached:>6}")
    return cached

print("Helpers ready — same 14k-token manual every call, only the first word changes")

Helpers ready — same 14k-token manual every call, only the first word changes


## Step 3 — Watch the Cache Hit

Send the same prefix a few times. The first call computes it fresh; by the next, the K/V are cached and reused.

In [33]:
import time

ask("Hello.", "When was Acme founded?")   # cold — computes the K/V
time.sleep(2)
ask("Hello.", "What is the return policy?")
hit = ask("Hello.", "What are the support hours?")
print(f"\n✅ HIT — {hit} tokens loaded from cache (the saved math, reused)")

first word: 'Hello.'   input:  14384  from cache:      0
first word: 'Hello.'   input:  14385  from cache:  14328
first word: 'Hello.'   input:  14385  from cache:  14328

✅ HIT — 14328 tokens loaded from cache (the saved math, reused)


## Step 4 — It's Keyed by Content, Not You

Same 14k manual, same question. Change only the **first word** — `Hello.` → `Hi.` — and the token sequence no longer matches. Watch the cache collapse.

In [34]:
miss = ask("Hi.", "What are the support hours?")
print(f"\n❌ MISS — {miss} from cache. One word changed → the prefix no longer matches.")

first word: 'Hi.'      input:  14385  from cache:      0

❌ MISS — 0 from cache. One word changed → the prefix no longer matches.


In [ ]:
print(f"Same first word  → {hit:>6} cached")
print(f"One word changed → {miss:>6} cached")
print("\nThe cache isn't keyed by 'you' or the session — it's keyed by the exact tokens.")

## Key Takeaways

1. **Tokens → K/V vectors is the expensive math** — caching stores those computed vectors, not your text
2. **A hit is decided by content** — the exact token prefix must match; change one word and you pay full price again
3. **So put the stable stuff first** — system prompt and context up top (cache-friendly), the changing question last. And remember: only the *input* is cached — the answer is always generated fresh.

---

Next: Day 41